# Passo 14 — Visualizações e EDA
**Dataset:** Oscar AMPAS — Winner Demographics (1927–2014)  
**Fonte:** Banco PostgreSQL (container oscar-db)

In [ ]:
import sys
!'{sys.executable}' -m pip install psycopg2-binary pandas matplotlib -q

In [ ]:
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

DB = dict(host='localhost', port=5432, dbname='oscar', user='postgres', password='brasil123')

def query(sql):
    with psycopg2.connect(**DB) as conn:
        return pd.read_sql(sql, conn)

print('Conexão OK')

## P1 — Filmes que varreram mais categorias

In [ ]:
df1 = query("""
    SELECT f.titulo, e.ano, COUNT(*) AS categorias_vencidas
    FROM premio p
    JOIN filme    f ON f.id_filme     = p.id_filme
    JOIN edicao   e ON e.id_edicao    = p.id_edicao
    JOIN categoria c ON c.id_categoria = p.id_categoria
    GROUP BY f.titulo, e.ano
    HAVING COUNT(*) > 1
    ORDER BY categorias_vencidas DESC, e.ano
    LIMIT 15
""")

df1['label'] = df1['titulo'] + ' (' + df1['ano'].astype(str) + ')'

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(df1['label'], df1['categorias_vencidas'], color='goldenrod')
ax.bar_label(bars, padding=3)
ax.set_xlabel('Categorias vencidas')
ax.set_title('Top 15 filmes com mais categorias vencidas na mesma edição')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('grafico_p1_filmes_multivencedores.png', dpi=110)
plt.show()

## P2 — Evolução da diversidade étnica por década

In [ ]:
df2 = query("""
    SELECT
        (e.ano / 10) * 10 AS decada,
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE v.etnia <> 'White') AS nao_brancos,
        ROUND(COUNT(*) FILTER (WHERE v.etnia <> 'White') * 100.0 / COUNT(*), 1) AS pct
    FROM premio p
    JOIN vencedor v ON v.id_vencedor = p.id_vencedor
    JOIN edicao   e ON e.id_edicao   = p.id_edicao
    GROUP BY decada ORDER BY decada
""")

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.bar(df2['decada'], df2['total'],     width=7, color='lightsteelblue', label='Total de prêmios')
ax1.bar(df2['decada'], df2['nao_brancos'], width=7, color='steelblue',   label='Não-brancos')
ax2.plot(df2['decada'], df2['pct'], color='crimson', marker='o', linewidth=2, label='% não-brancos')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())

ax1.set_xlabel('Década')
ax1.set_ylabel('Quantidade de prêmios')
ax2.set_ylabel('% não-brancos')
ax1.set_title('Diversidade étnica dos vencedores do Oscar por década')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig('grafico_p2_diversidade_decada.png', dpi=110)
plt.show()

## P3 — Idade dos vencedores por categoria

In [ ]:
df3 = query("""
    SELECT c.nome AS categoria,
           ROUND(AVG(e.ano - v.ano_nascimento), 1) AS idade_media,
           MIN(e.ano - v.ano_nascimento) AS mais_jovem,
           MAX(e.ano - v.ano_nascimento) AS mais_velho
    FROM premio p
    JOIN vencedor  v ON v.id_vencedor  = p.id_vencedor
    JOIN edicao    e ON e.id_edicao    = p.id_edicao
    JOIN categoria c ON c.id_categoria = p.id_categoria
    WHERE v.ano_nascimento IS NOT NULL
    GROUP BY c.nome ORDER BY idade_media
""")

fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(df3))
ax.bar(x, df3['idade_media'], color='coral', label='Idade média')
ax.errorbar(x,
            df3['idade_media'],
            yerr=[df3['idade_media'] - df3['mais_jovem'],
                  df3['mais_velho']  - df3['idade_media']],
            fmt='none', color='black', capsize=6, linewidth=1.5, label='Min–Max')

ax.set_xticks(x)
ax.set_xticklabels(df3['categoria'], rotation=15, ha='right')
ax.set_ylabel('Idade no ano da premiação')
ax.set_title('Idade dos vencedores por categoria (média e amplitude)')
ax.legend()
plt.tight_layout()
plt.savefig('grafico_p3_idade_categoria.png', dpi=110)
plt.show()
df3

## P4 — Maior intervalo entre vitórias consecutivas

In [ ]:
df4 = query("""
    WITH vitorias AS (
        SELECT v.nome, e.ano,
               LAG(e.ano) OVER (PARTITION BY v.id_vencedor ORDER BY e.ano) AS ano_anterior
        FROM premio p
        JOIN vencedor v ON v.id_vencedor = p.id_vencedor
        JOIN edicao   e ON e.id_edicao   = p.id_edicao
    )
    SELECT nome, ano_anterior AS primeiro, ano AS segundo,
           (ano - ano_anterior) AS intervalo
    FROM vitorias
    WHERE ano_anterior IS NOT NULL
    ORDER BY intervalo DESC LIMIT 10
""")

fig, ax = plt.subplots(figsize=(10, 5))
df4['label'] = df4['nome'] + ' (' + df4['primeiro'].astype(str) + '→' + df4['segundo'].astype(str) + ')'
bars = ax.barh(df4['label'], df4['intervalo'], color='mediumseagreen')
ax.bar_label(bars, fmt='%d anos', padding=3)
ax.set_xlabel('Intervalo (anos)')
ax.set_title('Top 10 maiores intervalos entre vitórias consecutivas')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('grafico_p4_intervalo_vitorias.png', dpi=110)
plt.show()

## P5 — Primeiro vencedor não-branco por categoria

In [ ]:
df5 = query("""
    WITH ranking AS (
        SELECT c.nome AS categoria, v.nome AS vencedor, v.etnia, e.ano,
               ROW_NUMBER() OVER (PARTITION BY c.id_categoria ORDER BY e.ano) AS rn
        FROM premio p
        JOIN vencedor  v ON v.id_vencedor  = p.id_vencedor
        JOIN edicao    e ON e.id_edicao    = p.id_edicao
        JOIN categoria c ON c.id_categoria = p.id_categoria
        WHERE v.etnia <> 'White'
    )
    SELECT categoria, vencedor, etnia, ano
    FROM ranking WHERE rn = 1 ORDER BY ano
""")

cores = {'Black': '#2c7bb6', 'Hispanic': '#d7191c', 'Asian': '#1a9641', 'Multiracial': '#fdae61'}

fig, ax = plt.subplots(figsize=(10, 4))
for _, row in df5.iterrows():
    cor = cores.get(row['etnia'], 'gray')
    ax.scatter(row['ano'], row['categoria'], color=cor, s=180, zorder=3)
    ax.annotate(f"{row['vencedor']}\n({row['etnia']})",
                xy=(row['ano'], row['categoria']),
                xytext=(6, 0), textcoords='offset points',
                fontsize=8, va='center')

ax.set_xlabel('Ano')
ax.set_title('Primeiro vencedor não-branco por categoria')
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('grafico_p5_primeira_vitoria_nao_branca.png', dpi=110)
plt.show()
df5